In [30]:
!pip install pygad

In [31]:
import numpy as np
import pygad
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [32]:
data = load_iris()
X = data.data
y = data.target.reshape(-1, 1) 

In [33]:
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [35]:
# Normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [46]:
help(pygad.GA)

Help on class GA in module pygad.pygad:

class GA(pygad.utils.parent_selection.ParentSelection, pygad.utils.crossover.Crossover, pygad.utils.mutation.Mutation, pygad.utils.nsga2.NSGA2, pygad.utils.validation.Validation, pygad.utils.engine.GAEngine, pygad.helper.unique.Unique, pygad.helper.misc.Helper, pygad.visualize.plot.Plot)
 |  GA(
 |      num_generations,
 |      num_parents_mating,
 |      fitness_func,
 |      fitness_batch_size=None,
 |      initial_population=None,
 |      sol_per_pop=None,
 |      num_genes=None,
 |      init_range_low=-4,
 |      init_range_high=4,
 |      gene_type=<class 'float'>,
 |      parent_selection_type='sss',
 |      keep_parents=-1,
 |      keep_elitism=1,
 |      K_tournament=3,
 |      crossover_type='single_point',
 |      crossover_probability=None,
 |      mutation_type='random',
 |      mutation_probability=None,
 |      mutation_by_replacement=False,
 |      mutation_percent_genes='default',
 |      mutation_num_genes=None,
 |      random_m

In [62]:
def fitness_fun(ga_instance, solution, solution_idx):
    num_neurons = int(solution[0])
    learning_rate = solution[1]

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(num_neurons, activation='relu', input_shape=(4,)),
        tf.keras.layers.Dense(3, activation='softmax')
    ])    

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=8, verbose=0)

    loss, accuracy = model.evaluate(X_train, y_train, verbose=0)

    return accuracy

In [63]:
gene_space = [range(5, 101), {'low': 0.0001, 'high': 0.1}]

In [69]:
ga_instance = pygad.GA(
    num_generations=3,
    num_parents_mating=2,
    fitness_func=fitness_fun,
    sol_per_pop=4,
    num_genes=2,
    gene_space=gene_space
)

In [70]:
print("Starting GA-NN Optimization...")
ga_instance.run()

Starting GA-NN Optimization...


In [71]:
solution, solution_fitness, solution_idx, = ga_instance.best_solution()

In [73]:
print(f"\n--- Best Parameters Found ---")
print(f"Optimal Neurons: {int(solution[0])}")
print(f"Optimal Learning Rate: {solution[1]:.4f}")
print(f"Best Accuracy Achieved: {solution_fitness * 100}%")


--- Best Parameters Found ---
Optimal Neurons: 81
Optimal Learning Rate: 0.0728
Best Accuracy Achieved: 98.33333492279053%
